In [24]:
#!pip install sentence_transformers
#!pip install ag2[retrievechat]
#!pip install pyautogen==0.7.2
#https://microsoft.github.io/autogen/0.2/docs/notebooks/agentchat_RetrieveChat/

# Setting base for Retrive Chat

In [25]:
import json
import os

import chromadb

import autogen
from autogen import AssistantAgent
from autogen.agentchat.contrib.retrieve_user_proxy_agent import RetrieveUserProxyAgent

# Accepted file formats for that can be stored in
# a vector database instance
from autogen.retrieve_utils import TEXT_FORMATS


with open("OAI_CONFIG_LIST.json", "r") as f:
    config_list = json.load(f)

assert len(config_list) > 0
print("models to use: ", [config_list[i]["model"] for i in range(len(config_list))])

models to use:  ['gpt-35-turbo', 'gpt-4-vision-preview', 'dalle', 'gemini-2.5-flash', 'gemini-1.5-pro-001', 'gemini-1.5-pro', 'gemini-pro-vision']


In [26]:
print("Accepted file formats for `docs_path`:")
print(TEXT_FORMATS)

Accepted file formats for `docs_path`:
['txt', 'json', 'csv', 'tsv', 'md', 'html', 'htm', 'rtf', 'rst', 'jsonl', 'log', 'xml', 'yaml', 'yml', 'pdf', 'mdx']


In [ ]:
config_list = [
    {
        "model": "gemini-3.5-flash",
        "api_key": "",
        "api_type": "google"
    }
]

In [28]:
import agentops
agentops.init("1e6d7ef3-23e9-4f57-b56b-b9eb489a6034")

In [29]:
# 1. create an AssistantAgent instance named "assistant"
assistant = AssistantAgent(
    name="assistant",
    system_message="You are a helpful assistant.",
    llm_config={
        "timeout": 600,
        "cache_seed": 42,
        "config_list": config_list,
    },
)


In [30]:
from autogen.agentchat.contrib.retrieve_user_proxy_agent import RetrieveUserProxyAgent

# Extract text from local PDF files
pdf_files = ["book.pdf"]
#docs_content = [extract_text_from_pdf(file) for file in pdf_files]

from autogen.retrieve_utils import TEXT_FORMATS,extract_text_from_pdf
docs_content= extract_text_from_pdf("book.pdf")

In [31]:
# Define the RetrieveUserProxyAgent instance
ragproxyagent = RetrieveUserProxyAgent(
    name="ragproxyagent",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=3,
    retrieve_config={
        "task": "qa",
        "docs_path": ["autogen-docs/book.pdf"],
        "docs_content": docs_content,  # Provide the extracted content directly
        "chunk_token_size": 200,
        "model": config_list[0]["model"],
        "vector_db": "chroma",
        "overwrite": True,
        "get_or_create": True,
    },
    code_execution_config=False,
)

# Process Starts for Scenario 1 : Generate code based off docstrings w/o human feedback

In [32]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

qa_problem = "summarize the content and use python code to any calculation given in the book"
chat_result = ragproxyagent.initiate_chat(assistant, message=ragproxyagent.message_generator, problem=qa_problem)
#agentops.end_session("Success")

Trying to create collection.


	The 22 ...
2026-07-07 10:54:09,523 - autogen.agentchat.contrib.retrieve_user_proxy_agent - INFO - Found 215 chunks.
2026-07-07 10:54:09,527 - autogen.agentchat.contrib.vectordb.chromadb - INFO - No content embedding is provided. Will use the VectorDB's embedding function to generate the content embedding.


VectorDB returns doc_ids:  [['fe1105cf', '8bf7f8ee', '1d0894aa', '839a2e04', 'd11d708f', '150c596a', 'fbda2b4c', '59e93e46', '1613971f', '25edaaee', '7091167f', 'c1c95a95', '43a28a91', '16a1e2eb', 'e7d73411', 'a5e9b671', 'a1061522', '05dac356', '1097b211', 'd41e75dd']]
Adding content of doc fe1105cf to context.


Adding content of doc 8bf7f8ee to context.


Adding content of doc 1d0894aa to context.


...

Adding content of doc e75197c0 to context.


Adding content of doc 54fb1eca to context.


Adding content of doc a683764c to context.


ragproxyagent (to assistant):

You're a retrieve augmented chatbot. You answer user's questions based on your own knowledge and the
context provided by the user.
If you can't answer the question with or without the current context, you should reply exactly `UPDATE CONTEXT`.
You must give as short an answer as possible.

User's question is: summarize the content and use python code to any calculation given in the book

Context is: ment with your prospect’ s mind. If the negative doesn’t
register quickly, your prospect will be confused and will
wonder, “What’ s this all about?”
Next, you have to shift quickly to the positive. The pur-
pose of candor isn’t to apologize. The purpose of candor is
to set up a benefit that will convince your prospect.
This law only proves the old maxim: Honesty is the best
policy.
91
THE LA W OF CANDOR
22 Laws of Marketing  10/31/02  12:23 PM  Page 9116
The Law of 
Singularity
In each situation, only one move
will produce substantial results.
92
22 Laws of Ma

assistant (to ragproxyagent):

**Summary:** 
"The 22 Immutable Laws of Marketing" by Al Ries and Jack Trout outlines fundamental rules for business success, emphasizing that marketing is a battle of perception, not products. Key concepts include being first in mind (Leadership/Mind), focusing on a single attribute (Focus), and navigating market division and hierarchy (Duality/Ladder).

**Python Calculations:**
Here are calculations based on the statistics provided in the text (Domino's total market size and the net number of car companies after 10 years from 1904):

```python
# Calculation 1: Total home delivery market size based on Domino's 4% share ($2.65 billion)
dominos_revenue_billions = 2.65
dominos_share = 0.04
total_delivery_market = dominos_revenue_billions / dominos_share

# Calculation 2: Remaining car companies 10 years after 1904
initial_companies = 60
companies_formed = 531
companies_perished = 346
remaining_companies = initial_companies + companies_formed - companies_per

In [33]:
#agentops.end_session("Success")

# Process Starts for Scenario 2 : Answer a question based off docstrings w/o human feedback

In [34]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

qa_problem = "You are a creative head . do many brainstorms"
chat_result = ragproxyagent.initiate_chat(assistant, message=ragproxyagent.message_generator, problem=qa_problem)

VectorDB returns doc_ids:  [['1097b211', '7e1ac0cd', '0e08a976', '81d8712b', 'be843ac3', 'a5e9b671', 'b2f1dbea', 'ba9a27e8', '21ec236c', '8126c05a', 'cf32340a', '7e3514a1', '89fd1dac', '5b19ade7', '10e789b5', '0e403d83', '71a3bd57', '1e7668a9', 'a1f246ff', '4cf44936']]
Adding content of doc 1097b211 to context.


Adding content of doc 7e1ac0cd to context.


Adding content of doc 0e08a976 to context.


Adding content of doc 81d8712b to context.


Adding content of doc be843ac3 to context.


...

Adding content of doc 5b19ade7 to context.


Adding content of doc 10e789b5 to context.


Adding content of doc 0e403d83 to context.


ragproxyagent (to assistant):

You're a retrieve augmented chatbot. You answer user's questions based on your own knowledge and the
context provided by the user.
If you can't answer the question with or without the current context, you should reply exactly `UPDATE CONTEXT`.
You must give as short an answer as possible.

User's question is: You are a creative head . do many brainstorms

Context is: ing is a long-term trend. 123
THE LA W OF ACCELERATION
22 Laws of Marketing  10/31/02  12:23 PM  Page 12322
The Law of 
Resources
Without adequate funding
an idea won’t get off the ground.
124
22 Laws of Marketing  10/31/02  12:23 PM  Page 124If you have a good idea and you’ve picked up this
book with the thought in mind that all you need is a lit-
tle marketing help, this chapter will throw cold water
on that thought.
Even the best idea in the world won’t go very far
without the money to get it off the ground. Inventors,
entrepreneurs, and assorted idea generators seem to
think that all thei

assistant (to ragproxyagent):

No, haphazardly generating ideas is not an efficient way to work.

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (a80d6f92-856a-4f40-9ce3-ba6cd1b2db00): Termination message condition on agent 'ragproxyagent' met


# Process Starts for Scenario 3 : Generate code based off docstrings w/ human feedback

In [35]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

# set `human_input_mode` to be `ALWAYS`, so the agent will ask for human input at every step.
ragproxyagent.human_input_mode = "ALWAYS"
code_problem = "Based on this book give Marketing idea for AI and Datascience trainig institute. Give 5 ideas. if any coding requires use python and save it as .py"
chat_result = ragproxyagent.initiate_chat(assistant, message=ragproxyagent.message_generator, problem=code_problem)

VectorDB returns doc_ids:  [['1097b211', '81d8712b', '89fd1dac', '0e08a976', '71a3bd57', '150c596a', 'be843ac3', 'ba9a27e8', '8bf7f8ee', '8a2a101c', '039e8163', 'd0e14ddf', 'b210f301', 'c1b21496', 'd5a791b1', 'a5e9b671', 'fc6e8864', '0e403d83', 'ef80c94d', '08e00728']]
Adding content of doc 1097b211 to context.


Adding content of doc 81d8712b to context.


Adding content of doc 89fd1dac to context.


Adding content of doc 0e08a976 to context.


Adding content of doc 71a3bd57 to context.


...

assistant (to ragproxyagent):

Address their perceptions directly, be completely honest about your limitations (candor), and keep your message simple.

--------------------------------------------------------------------------------
ragproxyagent (to assistant):

how to make good deals without a fight>

--------------------------------------------------------------------------------


assistant (to ragproxyagent):

Align with the prospect's existing perceptions and use candor (admitting a limitation) to build immediate trust.

--------------------------------------------------------------------------------
ragproxyagent (to assistant):

Any tips on gaining the customer's trust?

--------------------------------------------------------------------------------


assistant (to ragproxyagent):

Use candor: admit a negative first to make your positive claims instantly believable.

--------------------------------------------------------------------------------
ragproxyagent (to assistant):

How to make deals last longer?

--------------------------------------------------------------------------------


assistant (to ragproxyagent):

Focus on long-term trends rather than short-term fads, and consistently deliver on your core promise.

--------------------------------------------------------------------------------
ragproxyagent (to assistant):

Is marketing a easy job?

--------------------------------------------------------------------------------


assistant (to ragproxyagent):

No, because changing customer perceptions and following strict marketing laws is highly difficult.

--------------------------------------------------------------------------------

>>>>>>>> NO HUMAN INPUT RECEIVED.

>>>>>>>> TERMINATING RUN (f7b97817-9da8-451f-8bf3-3b16597f23ed): Termination message condition on agent 'ragproxyagent' met


# Process Starts for Scenario 4: Answer a question based off docstrings w/ human feedback

In [37]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

# set `human_input_mode` to be `ALWAYS`, so the agent will ask for human input at every step.
ragproxyagent.human_input_mode = "ALWAYS"
qa_problem = "Based on this book give Marketing idea for AI and Datascience trainig institute. Give 5 ideas. "
chat_result = ragproxyagent.initiate_chat(
    assistant, message=ragproxyagent.message_generator, problem=qa_problem
)  # type "exit" to exit the conversation

VectorDB returns doc_ids:  [['1097b211', '81d8712b', '71a3bd57', '89fd1dac', 'be843ac3', '150c596a', 'b210f301', '8a2a101c', 'ba9a27e8', '0e08a976', '0e403d83', '788184ee', '039e8163', 'a1f246ff', 'd5a791b1', '08e00728', '083871f5', 'c1b21496', '0c4927c1', '8cc896cd']]
Adding content of doc 1097b211 to context.


Adding content of doc 81d8712b to context.


Adding content of doc 71a3bd57 to context.


Adding content of doc 89fd1dac to context.


Adding content of doc be843ac3 to context.


...

Adding content of doc c1b21496 to context.


ragproxyagent (to assistant):

You're a retrieve augmented chatbot. You answer user's questions based on your own knowledge and the
context provided by the user.
If you can't answer the question with or without the current context, you should reply exactly `UPDATE CONTEXT`.
You must give as short an answer as possible.

User's question is: Based on this book give Marketing idea for AI and Datascience trainig institute. Give 5 ideas. 

Context is: ing is a long-term trend. 123
THE LA W OF ACCELERATION
22 Laws of Marketing  10/31/02  12:23 PM  Page 12322
The Law of 
Resources
Without adequate funding
an idea won’t get off the ground.
124
22 Laws of Marketing  10/31/02  12:23 PM  Page 124If you have a good idea and you’ve picked up this
book with the thought in mind that all you need is a lit-
tle marketing help, this chapter will throw cold water
on that thought.
Even the best idea in the world won’t go very far
without the money to get it off the ground. Inventors,
entrepreneurs, and as

assistant (to ragproxyagent):

1. **The Law of Leadership:** Be the first institute to specialize in a highly specific niche, like "Generative AI for Healthcare."
2. **The Law of Focus:** Own a single, simple word in the prospect's mind, such as "Employability."
3. **The Law of the Ladder:** If you are not the market leader, acknowledge it (e.g., "We are smaller, so we give more 1-on-1 mentorship").
4. **The Law of Perception:** Market positive student outcomes and prestige to build perception, rather than just listing curriculum features.
5. **The Law of Resources:** Secure a strong launch budget to heavily fund advertising, as ideas need financial backing to gain traction.

--------------------------------------------------------------------------------
ragproxyagent (to assistant):

Explain one law

--------------------------------------------------------------------------------


assistant (to ragproxyagent):

**The Law of Focus** states that a business succeeds by narrowing its focus to own a single, simple word or concept in the customer's mind (e.g., Federal Express owning "overnight").

--------------------------------------------------------------------------------
ragproxyagent (to assistant):

give me more info and example on that law

--------------------------------------------------------------------------------


assistant (to ragproxyagent):

**The Law of Focus** means narrowing your strategy to own a single, simple dictionary word in the prospect's mind, which requires sacrificing other product lines or messages to "burn" that one concept into their brain.

**Example:** Federal Express sacrificed its broad cargo services to focus on and successfully own the word "overnight."

--------------------------------------------------------------------------------

>>>>>>>> NO HUMAN INPUT RECEIVED.

>>>>>>>> TERMINATING RUN (2d811084-8139-4a4f-bfcd-65ad6d00a107): Termination message condition on agent 'ragproxyagent' met
